# 🫀 nnU-Net v2 end-to-end on MSD Task02_Heart — "Hello nnU-Net"

**Goal:** learn the complete nnU-Net v2 pipeline on the smallest Medical Segmentation
Decathlon task (left-atrium segmentation, 20 cases of 3D cardiac MRI), inside a single
free **Kaggle GPU session**.

| The 5 pipeline steps | Command |
|---|---|
| 1. Dataset in nnU-Net format | `Dataset002_Heart/` with `dataset.json` |
| 2. Fingerprint + planning + preprocessing | `nnUNetv2_plan_and_preprocess` |
| 3. Training | `nnUNetv2_train` |
| 4. Inference | `nnUNetv2_predict` |
| 5. Evaluation | `python -m src.metrics` (this repo) |

**⚙️ Before running:** Notebook Settings → Accelerator → **GPU T4 x2** (or P100), and
**Internet: On** (needed for pip + the Google Drive download).

**⏱️ Runtime budget:** install ~10 min · download ~10 min · planning/preprocessing ~5 min ·
training (100 epochs) ~3–6 h on T4 · predict+eval ~15 min. Fits one 12 h session with margin.

> Full 1000-epoch nnU-Net training is the SOTA recipe (~2 days on a good GPU). Here we use the
> built-in `nnUNetTrainer_100epochs` preset so everything fits in one session. The numbers are
> therefore *not* leaderboard numbers — the **pipeline understanding** is the point, and the same
> reduced budget is given to the Swin UNETR arm later, keeping the comparison fair.

## 0 · Install & environment

In [ ]:
!nvidia-smi -L
!pip install -q -U nnunetv2 gdown

In [ ]:
import os, json, glob, shutil
BASE = "/kaggle/working/nnunet_env"
for k in ("raw", "pre", "res"):
    os.makedirs(f"{BASE}/{k}", exist_ok=True)
os.environ["nnUNet_raw"]          = f"{BASE}/raw"
os.environ["nnUNet_preprocessed"] = f"{BASE}/pre"
os.environ["nnUNet_results"]      = f"{BASE}/res"
# keeps a CPU-backed Kaggle kernel from spawning 12 dataloader workers
os.environ.setdefault("nnUNet_n_proc_DA", "2")
print("nnUNet_raw          =", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      =", os.environ["nnUNet_results"])

In [ ]:
# Clone the repo (has src/metrics.py used for evaluation).
# >>> Replace with YOUR repo URL after you push it to GitHub <<<
!git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo
# If the repo is still private: upload it as a Kaggle Dataset instead and set REPO_DIR below.
REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)
print(os.listdir("."))

## 1 · Download MSD Task02_Heart (~2 GB, Google Drive)

The Medical Segmentation Decathlon is hosted on Google Drive; `gdown` fetches it.
(Off-Kaggle you can do the same with `python data/download_msd.py --task Task02_Heart`.)

In [ ]:
%%time
import gdown, tarfile, pathlib
TAR = "/kaggle/working/Task02_Heart.tar"
gdown.download(id="1wEB2I6S6tQBVEPxir8cB7kFB8gTQiQiT", path=TAR, quiet=False, fuzzy=True)
with tarfile.open(TAR) as tf:
    try:
        tf.extractall("/kaggle/working", filter="data")
    except TypeError:          # Python < 3.12
        tf.extractall("/kaggle/working")
task = pathlib.Path("/kaggle/working/Task02_Heart")
print("cases:", len(list((task/"imagesTr").glob("*.nii.gz"))))

## 2 · Convert to nnU-Net v2 format

The MSD folder *looks* like an nnU-Net dataset, but its `dataset.json` uses the **v1 key
`modality`**, while nnU-Net v2 hard-requires **`channel_names`** — copying it as-is makes
`plan_and_preprocess` fail at the integrity check. The repo's converter rewrites
`dataset.json` in the v2 schema (`channel_names`, `file_ending`, consecutive labels) and
copies the 20 cases into `Dataset002_Heart`.

In [ ]:
%%time
# The repo was cloned in the setup cell; the converter rewrites dataset.json in
# the v2 schema (channel_names instead of MSD's v1 'modality').
!cd {REPO_DIR} && python data/convert_to_nnunet.py --task heart --raw /kaggle/working/Task02_Heart --out $nnUNet_raw --copy

dj = json.load(open(os.path.join(os.environ["nnUNet_raw"], "Dataset002_Heart", "dataset.json")))
print("channels:", dj["channel_names"])
print("labels:  ", dj["labels"])
print("numTraining:", dj["numTraining"])

## 3 · Experiment planning & preprocessing

This is where nnU-Net's magic happens: it extracts a **dataset fingerprint** (image sizes,
voxel spacings, intensity statistics) and *automatically* decides patch size, batch size,
target spacing, normalization scheme, network depth and more. Watch what it chooses —
reading `nnUNetPlans.json` afterwards is one of the best ways to understand the framework.

In [ ]:
%%time
!nnUNetv2_plan_and_preprocess -d 2 --verify_dataset_integrity

In [ ]:
# 🔍 Inspect what nnU-Net planned for us
import json, glob
plans_path = glob.glob(os.path.join(os.environ["nnUNet_preprocessed"], "Dataset002_Heart", "nnUNetPlans.json"))[0]
plans = json.load(open(plans_path))
cfg = plans["configurations"]["3d_fullres"]
for key in ("patch_size", "batch_size", "spacing", "normalization_schemes", "n_conv_per_stage_encoder", "n_conv_per_stage_decoder"):
    print(f"{key:>28} : {cfg.get(key)}")
print("\nAvailable configurations:", list(plans["configurations"].keys()))

## 4 · Training (fold 0, 3d_fullres, 100-epoch preset)

nnU-Net's default is 1000 "epochs" (250 iterations each) with 5-fold cross-validation.
For a Kaggle session we train **fold 0 only, 100 epochs** via the built-in reduced trainer.
`--npz` additionally stores softmax outputs of the final validation (needed later for
ensembling / `find_best_configuration`).

📈 While it runs, you can open `progress.png` inside
`$nnUNet_results/Dataset002_Heart/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0/`
from the Kaggle file panel — it updates after every epoch.

In [ ]:
%%time
!nnUNetv2_train 2 3d_fullres 0 -tr nnUNetTrainer_100epochs --npz

## 5 · Inference on the **validation** cases

Only the 4 held-out cases of fold 0. Predicting the 16 training cases as well would make the
evaluation look better than the model is; the held-out slice is the number worth quoting, and
it is the same protocol the BraTS arm uses later.

In [ ]:
%%time
RAW = os.path.join(os.environ["nnUNet_raw"], "Dataset002_Heart")
PRED_DIR = "/kaggle/working/predictions/heart_fold0"
VAL_IN   = "/kaggle/working/val_images_heart"
splits = json.load(open(os.path.join(os.environ["nnUNet_preprocessed"], "Dataset002_Heart", "splits_final.json")))
val_cases = splits[0]["val"]
os.makedirs(VAL_IN, exist_ok=True); os.makedirs(PRED_DIR, exist_ok=True)
for c in val_cases:
    for f in glob.glob(os.path.join(RAW, "imagesTr", f"{c}_*.nii.gz")):
        shutil.copy2(f, VAL_IN)
print("held-out cases:", val_cases)
!nnUNetv2_predict -i {VAL_IN} -o {PRED_DIR} -d 2 -c 3d_fullres -f 0
print(sorted(os.listdir(PRED_DIR)))

## 6 · Evaluate with this repo's metric module

Dice + HD95 (in mm!) on the 4 held-out volumes — the exact same code path the BraTS
comparison uses, so the two arms are not measured with two different rulers.

In [ ]:
%%time
import sys; sys.path.insert(0, REPO_DIR)
from src.metrics import evaluate_folders
from pathlib import Path
GT = os.path.join(RAW, "labelsTr")
report = evaluate_folders(PRED_DIR, GT, labels=(1,))
for k, v in report.mean.items():
    print(k, "->", {kk: round(vv, 4) for kk, vv in v.items()})
report.save(Path("/kaggle/working/results/heart_nnunet"))

In [ ]:
# nnU-Net's OWN validation summary — it should agree with the numbers above
import glob
summ = glob.glob(os.path.join(os.environ["nnUNet_results"], "Dataset002_Heart", "*", "fold_0", "validation", "summary.json"))
if summ:
    s = json.load(open(summ[0]))
    print("mean:", json.dumps(s.get("mean", {}), indent=2)[:1200])
    print("full summary at:", summ[0])

## 7 · Package outputs for persistence

Anything in `/kaggle/working` is saved when you **Save Version → Save & Run All**, and can be
attached to later notebooks as a Dataset. We copy the artifacts that matter (metrics, curves,
plans, a few predictions) and zip them — never the raw data.

In [ ]:
%%time
out = "/kaggle/working/outputs_heart"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out, exist_ok=True)
shutil.copytree("/kaggle/working/results/heart_nnunet", f"{out}/metrics", dirs_exist_ok=True)
for f in glob.glob(os.path.join(os.environ["nnUNet_results"], "Dataset002_Heart/**/progress.png"), recursive=True):
    shutil.copy2(f, f"{out}/progress.png")
for f in glob.glob(os.path.join(os.environ["nnUNet_results"], "Dataset002_Heart/**/validation/summary.json"), recursive=True):
    shutil.copy2(f, f"{out}/nnunet_validation_summary.json")
shutil.copy2(os.path.join(os.environ["nnUNet_preprocessed"], "Dataset002_Heart", "nnUNetPlans.json"),
             f"{out}/nnUNetPlans.json")
shutil.copytree(VAL_IN, f"{out}/val_images", dirs_exist_ok=True)
shutil.copytree(PRED_DIR, f"{out}/predictions_val", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/nnunet_heart_outputs", "zip", out)
print("📦 zipped -> outputs_heart.zip")

## ✅ Done — what you learned

1. nnU-Net v2 dataset format (`DatasetXXX_Name`, `dataset.json`, `_0000` channel suffixes)
2. How the **fingerprint → plans** step auto-configures patch size / batch size / spacing
3. Training with reduced-epoch trainers under a compute budget (and what `--npz` is for)
4. Sliding-window inference with `nnUNetv2_predict`
5. Independent evaluation (Dice + HD95) with this repo's `src/metrics.py`

**Next:** `02_nnunet_brats.ipynb` — same pipeline, real research question:
brain tumor segmentation on BraTS, with a shared data split for the Swin UNETR comparison.